In [3]:
import pandas as pd
import numpy as np
import random
import os
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, precision_recall_curve
from sklearn.model_selection import StratifiedKFold

from data_preprocessing.get_stnthetic_data import *

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(100)  # ✅ Set your global seed here

## Load dataset

In [4]:
data = pd.read_csv("/Users/ccy/Documents/CMU/Spring2025/42687 Projects in Biomedical AI/Final Project/Explainable Deep Learning on Multimodal Patient Data for Predicting Immunotherapy Outcomes in Metastatic Non-small Cell Lung Cancer/Synthetic data/PCA/rfe_50.csv")

X_train = data.iloc[:60, :-3].values
print(X_train.shape)
# Standardize the features
#scaler = StandardScaler()
#X_scaled = scaler.fit_transform(X_train)
target_col = data.columns[-3]
print(target_col)
y = data[target_col].values
y = y[:60]

print("Label distribution:", Counter(y))


X_test = data.iloc[60:, :-3].values
print(X_test.shape)
# Standardize the features
#scaler = StandardScaler()
#X_test = scaler.fit_transform(X_test)
target_col = data.columns[-3]
print(target_col)
y_test = data[target_col].values
y_test = y_test[60:]
print("Label distribution:", Counter(y_test))


(60, 50)
Best response
Label distribution: Counter({0.0: 31, 1.0: 29})
(13, 50)
Best response
Label distribution: Counter({1.0: 9, 0.0: 4})


### Define MLP

In [5]:
# Define MLP Model
class MLP(nn.Module):
    def __init__(self, input_size, hidden_dims=[16, 8, 4], dropout=0.3):
        super(MLP, self).__init__()
        layers = []
        in_dim = input_size
        for h in hidden_dims:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return torch.sigmoid(self.model(x))


### Training

In [6]:
def cross_validate_models(X_scaled, y, model_configs, num_epochs=100, batch_size=32):
    results = []

    for config in model_configs:
        print(f"\n🔍 Testing model config: {config}")

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        train_f1_scores, val_f1_scores = [], []
        train_acc_scores, val_acc_scores = [], []


        for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
            X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
            X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
            y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
            y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

            train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
            val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size, shuffle=False)

            model = MLP(input_size=X_scaled.shape[1], hidden_dims=config["hidden_dims"], dropout=config["dropout"])
            criterion = nn.BCELoss()
            optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)

            all_train_preds, all_train_labels = [], []
            for epoch in range(num_epochs):
                model.train()
                for inputs, labels in train_loader:
                    optimizer.zero_grad()
                    outputs = model(inputs)
                    preds = (outputs > 0.5).float()
                    all_train_preds.extend(preds.cpu().numpy())
                    all_train_labels.extend(labels.cpu().numpy())
                    
                    loss = criterion(outputs, labels)
                    loss.backward()
                    optimizer.step()
            
            train_f1 = f1_score(all_train_labels, all_train_preds)
            train_acc = accuracy_score(all_train_labels, all_train_preds)
            train_f1_scores.append(train_f1)
            train_acc_scores.append(train_acc)

            # Evaluation
            model.eval()
            all_val_preds, all_val_labels = [], []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    outputs = model(inputs)
                    preds = (outputs > 0.5).float()
                    all_val_preds.extend(preds.cpu().numpy())
                    all_val_labels.extend(labels.cpu().numpy())

            
            val_f1 = f1_score(all_val_labels, all_val_preds)
            val_acc = accuracy_score(all_val_labels, all_val_preds)
            val_f1_scores.append(val_f1)
            val_acc_scores.append(val_acc)
            
            print(f"Fold {fold+1}: Train F1={train_f1:.4f}, Val F1={val_f1:.4f}")

        config['avg_train_f1'] = np.mean(train_f1_scores)
        config['avg_val_f1'] = np.mean(val_f1_scores)
        config['avg_train_acc'] = np.mean(train_acc_scores)
        config['avg_val_acc'] = np.mean(val_acc_scores)
        
        print(f"✅ Config {config['hidden_dims']} → Train acc: {config['avg_train_acc']:.4f}, Val acc: {config['avg_val_acc']:.4f}")
        print(f"✅ Config {config['hidden_dims']} → Train F1: {config['avg_train_f1']:.4f}, Val F1: {config['avg_val_f1']:.4f}")

        results.append(config)

    return sorted(results, key=lambda x: x['avg_val_f1'], reverse=True)


In [7]:
def train_final_model(X_scaled, y, hidden_dims, dropout=0.3, num_epochs=100, batch_size=32, save_path="final_model.pt"):
    print("\n🚀 Training final model with best config...")
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    loader = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=True)

    model = MLP(input_size=X_scaled.shape[1], hidden_dims=hidden_dims, dropout=dropout)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)

    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    torch.save(model.state_dict(), save_path)
    print(f"✅ Final model saved to {save_path}")
    return model


In [8]:
def evaluate_on_test(model, X_test, y_test):
    model.eval()
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

    with torch.no_grad():
        outputs = model(X_test_tensor)
        preds = (outputs > 0.5).float().cpu().numpy()
        labels = y_test_tensor.cpu().numpy()

    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)
    f1 = f1_score(labels, preds)
    print(f"\n🧪 Final Evaluation on Test Set:")
    print(f"Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")


In [9]:
model_configs = [
    {"hidden_dims": [16, 8, 4], "dropout": 0.3},
    {"hidden_dims": [32, 16], "dropout": 0.3},
    {"hidden_dims": [64, 32], "dropout": 0.2},
    {"hidden_dims": [16, 8], "dropout": 0.3}
]

# Step 1: Select the best model
sorted_configs = cross_validate_models(X_train, y, model_configs)


🔍 Testing model config: {'hidden_dims': [16, 8, 4], 'dropout': 0.3}
Fold 1: Train F1=0.6288, Val F1=0.5455
Fold 2: Train F1=0.4857, Val F1=0.8333
Fold 3: Train F1=0.4738, Val F1=0.6667
Fold 4: Train F1=0.4960, Val F1=0.7273
Fold 5: Train F1=0.6105, Val F1=0.7143
✅ Config [16, 8, 4] → Train acc: 0.5906, Val acc: 0.7000
✅ Config [16, 8, 4] → Train F1: 0.5390, Val F1: 0.6974

🔍 Testing model config: {'hidden_dims': [32, 16], 'dropout': 0.3}
Fold 1: Train F1=0.7184, Val F1=0.5000
Fold 2: Train F1=0.6795, Val F1=0.5455
Fold 3: Train F1=0.6768, Val F1=0.5455
Fold 4: Train F1=0.6800, Val F1=0.6000
Fold 5: Train F1=0.7528, Val F1=0.5714
✅ Config [32, 16] → Train acc: 0.6937, Val acc: 0.5667
✅ Config [32, 16] → Train F1: 0.7015, Val F1: 0.5525

🔍 Testing model config: {'hidden_dims': [64, 32], 'dropout': 0.2}
Fold 1: Train F1=0.7508, Val F1=0.7692
Fold 2: Train F1=0.7675, Val F1=0.6000
Fold 3: Train F1=0.7641, Val F1=0.5455
Fold 4: Train F1=0.7187, Val F1=0.6667
Fold 5: Train F1=0.8159, Val F1

In [10]:
best_config = sorted_configs[0]
print(f"best config: {best_config}")

# Step 2: retrain best model on full training set
final_model = train_final_model(X_train, y, hidden_dims=best_config['hidden_dims'], dropout=best_config['dropout'])

# Step 3: test set
evaluate_on_test(final_model, X_test, y_test)


best config: {'hidden_dims': [16, 8, 4], 'dropout': 0.3, 'avg_train_f1': 0.5389661707732138, 'avg_val_f1': 0.6974025974025974, 'avg_train_acc': 0.5905833333333333, 'avg_val_acc': 0.7}

🚀 Training final model with best config...
✅ Final model saved to final_model.pt

🧪 Final Evaluation on Test Set:
Accuracy: 0.6923, Precision: 0.7778, Recall: 0.7778, F1: 0.7778


### Loss and accuracy visualization